<div style="text-align: center;">
  <img src="../docs/imgs/banner.png" width="500">
</div>

# Preference Fine-tuning with DPO

### Imports and Model/Data Loading

In this class we will preference fine-tune [**`HuggingFaceTB/SmolLM2-360M-Instruct`**](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct).

SmolLM2 is still the same **decoder-only transformer** we used in the LoRA notebook. The architecture does not change at all: after training, it will still generate normal natural language autoregressively.

What changes is the **training signal**. Instead of giving the model one target completion to imitate, we now give it **two candidate responses** and tell it which one humans preferred. We will use that pairwise preference information to fine-tune the model with **Direct Preference Optimisation (DPO)**.

In [ ]:
import yaml
from dotenv import load_dotenv; load_dotenv()
from finetuning._utils import setup_hf; setup_hf()

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset
import torch

from finetuning._defaults import WORKSHOP_DEFAULTS, ROOT_DIR
from finetuning._config import DPOConfig

In [ ]:
# Load tokenizer and model (this will take a while the first time)

tokenizer = AutoTokenizer.from_pretrained(WORKSHOP_DEFAULTS.model)
model = AutoModelForCausalLM.from_pretrained(WORKSHOP_DEFAULTS.model)

For this section of the workshop we will fine-tune the model using [**`stanfordnlp/SHP-2`**](https://huggingface.co/datasets/stanfordnlp/SHP-2). This is a large NLP dataset prepared by Stanford researchers, which uses QA-style pairs from Reddit.

Each row contains:
- a **history** field, which plays the role of the prompt
- two candidate responses, **`human_ref_A`** and **`human_ref_B`**
- a binary **label** telling us which response humans preferred (most upvotes)

So unlike the [`lora.ipynb`](lora.ipynb) notebook, the dataset is not already in a chat training format. We will preprocess it for you into a simpler `prompt / chosen / rejected` schema before training.

In [ ]:
# Load dataset

ds = load_dataset("stanfordnlp/SHP-2", split="train", streaming=True)

ds = ds.shuffle(seed=42, buffer_size=10_000)
ds = list(ds.take(1000))

In [ ]:
# Let's pick out a raw example

row = ds[37]

print("history:\n")
print(row["history"])
print("\n" + "-" * 80 + "\n")

print("human_ref_A:\n")
print(row["human_ref_A"])
print("\n" + "-" * 80 + "\n")

print("human_ref_B:\n")
print(row["human_ref_B"])
print("\n" + "-" * 80 + "\n")

print("labels:", row["labels"], "(1 means A is preferred, 0 means B is preferred)")

### DPO Configuration Loading

We are now going to load the configuration for our DPO experiment. The hyperparameters are set in a `yaml` configuration file, in exactly the same spirit as [`lora.ipynb`](lora.ipynb). We encourage everyone to experiment with these parameters to produce different results in their own training process, but we place some restrictions on possible values to keep computations tractable on the GPUs we are working with. The fields are as follows:

- **num_epochs**: number of epochs for training. *Allowed range: [1, 3]*.
- **learning_rate**: learning rate for training. *Allowed range: [0, ∞)*.
- **beta**: DPO strength parameter controlling how strongly the model is pushed toward preferred answers. *Allowed range: (0.0, 1.0]*.
- **lora_rank**: rank of the LoRA adapter to be trained. *Allowed range: [1, 32]*.
- **lora_alpha**: LoRA scaling parameter. *Allowed range: [1, ∞)*.
- **micro_batch_size**: number of preference pairs processed per step before any gradient accumulation. *Allowed range: [1, 4]*.
- **gradient_accumulation_steps**: number of steps to accumulate gradients before updating the model. *Allowed range: [1, 16]*.
- **target_modules**: list of linear layers to which LoRA adapters are applied.

In [ ]:
CONFIG_PATH = ROOT_DIR / "configs" / "example_dpo.yaml"  # Change example_dpo.yaml to your config file, or update it in place!

with open(CONFIG_PATH) as f:
    data = yaml.safe_load(f)

config = DPOConfig(**data)

config

When building your training loop, you'll access hyperparameters via the config object we've just instantiated, e.g.

In [ ]:
config.beta

### Training

Let's start by talking to our base model. Even after DPO, the model will still produce normal natural language, so it is useful to see a baseline response before we train.

In [ ]:
USER_MESSAGE = "Successful defense! Today I defended my PhD dissertation, and it was accepted by committee with revisions. I'm so relieved I could cry. I'm so glad this stage is over. To be honest, I had been worried the pandemic was going to affect my defense process negatively, and I'm so relieved it's done with."

messages = [
    {"role": "user", "content": USER_MESSAGE},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=WORKSHOP_DEFAULTS.max_new_tokens,
        temperature=WORKSHOP_DEFAULTS.temperature,
        do_sample=WORKSHOP_DEFAULTS.do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

base_response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

base_response

Now let's build the DPO fine-tuning pipeline step by step. The aim here is again to make the mechanics of training as visible as possible, and the sample solutions for the below exercises can be found in [`notebooks/solutions/dpo_solutions.ipynb`](solutions/dpo_solutions.ipynb).

We will:
1. Preprocess SHP-2 into prompt / chosen / rejected examples
2. Tokenize chosen and rejected continuations separately
3. Wrap the base model with LoRA adapters
4. Build a dataloader and optimizer
5. Write a minimal DPO training loop with gradient accumulation
6. Evaluate the tuned model with held-out preference accuracy

#### 1. Preprocess the dataset

SHP-2 does not come in the exact format we want, so we will preprocess it for you here.

For each row we will:
- Use `history` as the prompt
- Use the label to decide which response is `chosen`
- Use the other response as `rejected`
- Keep only a small subset so training stays workshop-sized

In [ ]:
ds = Dataset.from_list(ds)

def convert_shp_to_preference(example):
    prompt = example["history"]

    if example["labels"] == 1:
        chosen = example["human_ref_A"]
        rejected = example["human_ref_B"]
    else:
        chosen = example["human_ref_B"]
        rejected = example["human_ref_A"]

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

train_raw = ds.select(range(WORKSHOP_DEFAULTS.dpo_train_size)).map(convert_shp_to_preference)
test_raw = ds.select(range(WORKSHOP_DEFAULTS.dpo_train_size, WORKSHOP_DEFAULTS.dpo_train_size + WORKSHOP_DEFAULTS.dpo_test_size)).map(convert_shp_to_preference)

keep_columns = ["prompt", "chosen", "rejected"]
train_raw = train_raw.remove_columns([c for c in train_raw.column_names if c not in keep_columns])
test_raw = test_raw.remove_columns([c for c in test_raw.column_names if c not in keep_columns])

train_raw[37]

In [ ]:
# Let's inspect one processed example

row = train_raw[37]

print("prompt:\n")
print(row["prompt"])
print("\n" + "-" * 80 + "\n")

print("chosen:\n")
print(row["chosen"])
print("\n" + "-" * 80 + "\n")

print("rejected:\n")
print(row["rejected"])

#### 2. Tokenize the dataset

For DPO, we need to score the preferred and dispreferred continuations separately.

We therefore build two tokenized sequences for each example:
- `prompt + chosen`
- `prompt + rejected`

We also keep labels where:
- prompt tokens are set to `-100`
- response tokens keep their real token ids

This lets us score only the response continuation when we compute sequence log-probabilities.

In [ ]:
from peft import LoraConfig as PeftLoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader

def build_prompt_text(prompt):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )

def tokenize_preference_example(example):
    prompt_text = build_prompt_text(example["prompt"])
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=WORKSHOP_DEFAULTS.max_prompt_length,
        add_special_tokens=False,
    )

    chosen_tokens = tokenizer(
        example["chosen"],
        truncation=True,
        max_length=WORKSHOP_DEFAULTS.max_new_tokens,
        add_special_tokens=False,
    )

    rejected_tokens = tokenizer(
        example["rejected"],
        truncation=True,
        max_length=WORKSHOP_DEFAULTS.max_new_tokens,
        add_special_tokens=False,
    )

    chosen_input_ids = prompt_tokens["input_ids"] + chosen_tokens["input_ids"] + [tokenizer.eos_token_id]
    chosen_attention_mask = prompt_tokens["attention_mask"] + chosen_tokens["attention_mask"] + [1]
    chosen_labels = [-100] * len(prompt_tokens["input_ids"]) + chosen_tokens["input_ids"] + [tokenizer.eos_token_id]

    rejected_input_ids = prompt_tokens["input_ids"] + rejected_tokens["input_ids"] + [tokenizer.eos_token_id]
    rejected_attention_mask = prompt_tokens["attention_mask"] + rejected_tokens["attention_mask"] + [1]
    rejected_labels = [-100] * len(prompt_tokens["input_ids"]) + rejected_tokens["input_ids"] + [tokenizer.eos_token_id]

    return {
        "chosen_input_ids": chosen_input_ids,
        "chosen_attention_mask": chosen_attention_mask,
        "chosen_labels": chosen_labels,
        "rejected_input_ids": rejected_input_ids,
        "rejected_attention_mask": rejected_attention_mask,
        "rejected_labels": rejected_labels,
    }


train_ds = train_raw.map(tokenize_preference_example, remove_columns=train_raw.column_names)
test_ds = test_raw.map(tokenize_preference_example, remove_columns=test_raw.column_names)

train_ds[0]

#### 3. Attach LoRA adapters

Now wrap the base model with LoRA using the hyperparameters from `config`.

Just as before, only a small subset of weights should be trainable.

In [ ]:
peft_config = PeftLoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=________________,
    lora_alpha=________________,
    target_modules=________________,
    bias="none",
)

# TODO: wrap the base model with LoRA adapters
model = ________________________________

model.print_trainable_parameters()

#### 4. Build the dataloader and optimizer

We now need a small custom collator because every batch contains:
- one chosen sequence
- one rejected sequence

We will still use AdamW for optimisation.

In [ ]:
def collate_preference_batch(batch):
    chosen_features = [
        {
            "input_ids": item["chosen_input_ids"],
            "attention_mask": item["chosen_attention_mask"],
            "labels": item["chosen_labels"],
        }
        for item in batch
    ]
    rejected_features = [
        {
            "input_ids": item["rejected_input_ids"],
            "attention_mask": item["rejected_attention_mask"],
            "labels": item["rejected_labels"],
        }
        for item in batch
    ]

    chosen_batch = tokenizer.pad(chosen_features, padding=True, return_tensors="pt")
    rejected_batch = tokenizer.pad(rejected_features, padding=True, return_tensors="pt")

    return {
        "chosen_input_ids": chosen_batch["input_ids"],
        "chosen_attention_mask": chosen_batch["attention_mask"],
        "chosen_labels": chosen_batch["labels"],
        "rejected_input_ids": rejected_batch["input_ids"],
        "rejected_attention_mask": rejected_batch["attention_mask"],
        "rejected_labels": rejected_batch["labels"],
    }


# TODO: build the dataloader using the micro batch size from config
train_loader = ________________________________

# TODO: create the optimizer with config.learning_rate
optimizer = ________________________________

len(train_loader)

#### 5. Write the DPO helpers

This is the new technical ingredient.

We need a helper that turns model logits into a sequence log-probability over just the response tokens. Then we compare:
- chosen vs. rejected under the current policy
- chosen vs. rejected under the reference model

For the reference model we will use the same base model with LoRA adapters temporarily disabled, which is the cheapest memory setup.

In [ ]:
import torch.nn.functional as F

def sequence_logprob(logits, labels):
    # TODO: shift logits and labels for next token prediction
    shift_logits = ________________________________
    shift_labels = ________________________________

    # TODO: convert logits to log probabilities
    log_probs = ________________________________

    # TODO: gather token log probabilities for all non masked labels
    safe_labels = shift_labels.masked_fill(shift_labels == -100, 0)
    token_log_probs = ________________________________

    # TODO: zero out prompt positions and sum only response token scores
    mask = ________________________________
    seq_logprob = ________________________________

    return seq_logprob


def dpo_loss(model, batch, beta):
    chosen_outputs = model(
        input_ids=batch["chosen_input_ids"],
        attention_mask=batch["chosen_attention_mask"],
    )
    rejected_outputs = model(
        input_ids=batch["rejected_input_ids"],
        attention_mask=batch["rejected_attention_mask"],
    )

    # TODO: compute policy log probabilities
    policy_chosen_logps = ________________________________
    policy_rejected_logps = ________________________________

    with model.disable_adapter():
        ref_chosen_outputs = model(
            input_ids=batch["chosen_input_ids"],
            attention_mask=batch["chosen_attention_mask"],
        )
        ref_rejected_outputs = model(
            input_ids=batch["rejected_input_ids"],
            attention_mask=batch["rejected_attention_mask"],
        )

    # TODO: compute reference log probabilities
    ref_chosen_logps = ________________________________
    ref_rejected_logps = ________________________________

    # TODO: compute DPO logits and final loss
    logits = ________________________________
    loss = ________________________________

    return loss

#### 6. Write the training loop

This is the key exercise. Just as before, gradient accumulation lets us simulate a larger effective batch size:

`global_batch_size = micro_batch_size × gradient_accumulation_steps`

Fill in the blanks below.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.train()
optimizer.zero_grad()

loss_history = []

global_step = 0
for epoch in range(config.num_epochs):
    for step, batch in enumerate(train_loader, start=1):
        # TODO: move every tensor in the batch onto the device
        batch = ________________________________

        # TODO: compute the DPO loss and scale it for gradient accumulation
        loss = ________________________________

        # TODO: backpropagate
        ________________________________

        if step % config.gradient_accumulation_steps == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        loss_history.append(loss.item() * config.gradient_accumulation_steps)

        if step % 10 == 0 or step == len(train_loader):
            print(
                f"epoch={epoch + 1} step={step}/{len(train_loader)} "
                f"loss={loss_history[-1]:.4f}"
            )

---
Congratulations, you have just preference fine-tuned an LLM!

Now, let's see how our model does on the same prompt as before.

In [ ]:
USER_MESSAGE = "Successful defense! Today I defended my PhD dissertation, and it was accepted by committee with revisions. I'm so relieved I could cry. I'm so glad this stage is over. To be honest, I had been worried the pandemic was going to affect my defense process negatively, and I'm so relieved it's done with."

messages = [
    {"role": "user", "content": USER_MESSAGE},
]

model.eval()

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=WORKSHOP_DEFAULTS.max_new_tokens,
        temperature=WORKSHOP_DEFAULTS.temperature,
        do_sample=WORKSHOP_DEFAULTS.do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

finetuned_response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

finetuned_response

And let's evaluate preference accuracy on the test set:

In [ ]:
test_loader = DataLoader(
    test_ds,
    batch_size=config.micro_batch_size,
    shuffle=False,
    collate_fn=collate_preference_batch,
)

def compute_preference_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            chosen_outputs = model(
                input_ids=batch["chosen_input_ids"],
                attention_mask=batch["chosen_attention_mask"],
            )
            rejected_outputs = model(
                input_ids=batch["rejected_input_ids"],
                attention_mask=batch["rejected_attention_mask"],
            )

            chosen_logps = sequence_logprob(chosen_outputs.logits, batch["chosen_labels"])
            rejected_logps = sequence_logprob(rejected_outputs.logits, batch["rejected_labels"])

            correct += (chosen_logps > rejected_logps).sum().item()
            total += chosen_logps.size(0)

    return correct / total


print(f"Test preference accuracy: {compute_preference_accuracy(model, test_loader):.3f}")